# HW8 (20')

<font size='4'>

For this assignment, it is a combination of jupyter notebook assignment and python scripts.

For Q1, please upload your outputs including codes and graphics to your own GitHub repository. <br> You will need to disclose your GitHub repository below.

For Q2, please submit this jupyter notebook as an HTML or PDF file.

First of all, print your name (First and Last) below.

In [81]:
print("Victoria Ostebo")

Victoria Ostebo


## 0. Import relevant packages

In [82]:
import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.formula.api as smf

## Q1. Convert your HW7 to python scripts. (10')

<font size='4'>

- Under your working directory, there should be a folder called `self_py_fun`.
- Create a new python file called `HW8Fun.py` and move previously defined functions `produce_trun_mean_cov()`, `plot_trunc_mean()`, and `plot_trunc_cov()` to that file. Make sure you import proper packages.
- Create another main file `HW8_main.py`.
- Import relevant packages, modules, and/or function.
- Copy the global variables and call your functions inside `HW8_main.py`.
- A major difference compared to HW7 is that you are asked to save those figures to your local working environment.
    - Create a new directory `K114` under your current working directory.
    - For mean functions, please save it as a `Mean.png` output using `plt.savefig()` function.
    - The changes should be made within `HW8Fun.py` rather than `HW8_main.py`.
    - For covariance matrices, please save them as `Covariance_Target.png`, `Covariance_Non-Target.png`, and `Covariance_All.png` outputs using the same function above.
    - To summarize, there should be **four** figures under `K114` folder.
- Upload your entire work to your GitHub repository via push button.

In [83]:
# Provide your GitHub repository link below in the Markdown chunk. Remember to make it public and make the link clickable.
# Do not include sensitive information in your GitHub repository.


In [84]:
# Write your own code
!mkdir -p self_py_fun

In [85]:
%%writefile self_py_fun/HW8Fun.py
import numpy as np
import matplotlib.pyplot as plt

def produce_trunc_mean_cov(input_signal, input_type, E_val):
    input_type = np.squeeze(input_type)
    sample_size_len = input_signal.shape[0]
    feature_len = input_signal.shape[1]
    length_per_electrode = feature_len // E_val
    X = input_signal.reshape(sample_size_len, E_val, length_per_electrode)
    X_ntar = X[input_type == -1]
    X_tar = X[input_type == 1]
    signal_ntar_mean = np.mean(X_ntar, axis=0)
    signal_tar_mean = np.mean(X_tar, axis=0)
    signal_tar_cov = np.zeros((E_val, length_per_electrode, length_per_electrode))
    signal_ntar_cov = np.zeros((E_val, length_per_electrode, length_per_electrode))
    signal_all_cov = np.zeros((E_val, length_per_electrode, length_per_electrode))
    for e in range(E_val):
        signal_tar_cov[e] = np.cov(X_tar[:, e, :], rowvar=False, ddof=1)
        signal_ntar_cov[e] = np.cov(X_ntar[:, e, :], rowvar=False, ddof=1)
        signal_all_cov[e] = np.cov(X[:, e, :], rowvar=False, ddof=1)
    return [signal_tar_mean, signal_ntar_mean, signal_tar_cov, signal_ntar_cov, signal_all_cov]

def plot_trunc_mean(eeg_tar_mean, eeg_ntar_mean, subject_name, time_index, E_val, electrode_name_ls, save_dir, y_limit=np.array([-5, 8]), fig_size=(12, 12)):
    fig, axes = plt.subplots(4, 4, figsize=(12, 12))
    for i in range(16):
        row = i % 4
        col = i // 4
        ax = axes[row, col]
        ax.plot(time_index, eeg_tar_mean[i], color='red', label='Target')
        ax.plot(time_index, eeg_ntar_mean[i], color='blue', label='Non-Target')
        ax.set_xlabel('Time (ms)')
        ax.set_ylabel('Amplitude (muV)')
        ax.set_title(electrode_name_ls[i])
        if i == 0:  
            ax.legend()
    fig.suptitle(subject_name)
    plt.tight_layout()
    plt.savefig(f'{save_dir}/Mean.png')
    plt.close()

def plot_trunc_cov(eeg_cov, cov_type, time_index, subject_name, E_val, electrode_name_ls, save_dir, fig_size=(14, 12)):
    fig, axes = plt.subplots(4, 4, figsize=fig_size)
    fig.suptitle(f'{cov_type} - Subject: {subject_name}', fontsize=14, weight='bold')
    X, Y = np.meshgrid(time_index, time_index)
    for e in range(E_val):
        r = e % 4  
        c = e // 4
        ax = axes[r, c]
        cont = ax.contourf(X, Y, eeg_cov[e, :, :], cmap='plasma')
        ax.set_title(electrode_name_ls[e])
        ax.set_xlabel('Time (ms)')
        ax.set_ylabel('Time (ms)')
        ax.invert_yaxis()
    fig.colorbar(cont, ax=axes, orientation='vertical', fraction=0.03, pad=0.05)
    plt.tight_layout()
    plt.savefig(f'{save_dir}/Covariance_{cov_type}.png')
    plt.close()

Overwriting self_py_fun/HW8Fun.py


In [86]:
%%writefile HW8_main.py
import os
import numpy as np
import scipy.io as sio
import sys
sys.path.append('./self_py_fun')
from HW8Fun import produce_trunc_mean_cov, plot_trunc_mean, plot_trunc_cov

#global constants
bp_low = 0.5
bp_upp = 6
E_val = 16
electrode_name_ls = ['F3', 'Fz', 'F4', 'T7', 'C3', 'Cz', 'C4', 'T8','CP3', 'CP4', 'P3', 'Pz', 'P4', 'PO7', 'PO8', 'Oz']
subject_name = 'K114'
session_name = '001_BCI_TRN'

#load MATLAB file
mat_path = 'data/K114_001_BCI_TRN_Truncated_Data_0.5_6.mat'
eeg_trunc_obj = sio.loadmat(mat_path)
eeg_trunc_signal = eeg_trunc_obj['Signal']
eeg_trunc_type   = eeg_trunc_obj['Type']

#make K114 directory
save_directory = subject_name
os.makedirs(save_directory, exist_ok=True)
print(f"Output dir: {save_directory}")

#means & covariances
(signal_tar_mean, signal_ntar_mean,
 signal_tar_cov, signal_ntar_cov, signal_all_cov) = produce_trunc_mean_cov(eeg_trunc_signal, eeg_trunc_type, E_val)

#time axes
length_per_electrode = signal_tar_mean.shape[1]
time_index_mean = np.linspace(0, 800, length_per_electrode)
time_index_cov  = np.arange(length_per_electrode)

#plot & save 
plot_trunc_mean(
    eeg_tar_mean = signal_tar_mean,
    eeg_ntar_mean = signal_ntar_mean,
    subject_name = f"Subject {subject_name}",
    time_index = time_index_mean,
    E_val = E_val,
    electrode_name_ls = electrode_name_ls,
    save_dir = save_directory)

plot_trunc_cov(signal_tar_cov,  "Target",     time_index_cov, f"Subject {subject_name}", E_val, electrode_name_ls, save_directory)
plot_trunc_cov(signal_ntar_cov, "Non-Target", time_index_cov, f"Subject {subject_name}", E_val, electrode_name_ls, save_directory)
plot_trunc_cov(signal_all_cov,  "All",        time_index_cov, f"Subject {subject_name}", E_val, electrode_name_ls, save_directory)

print("Saved figures:",
      f"{save_directory}/Mean.png",
      f"{save_directory}/Covariance_Target.png",
      f"{save_directory}/Covariance_Non-Target.png",
      f"{save_directory}/Covariance_All.png", sep="\n")

Overwriting HW8_main.py


In [87]:
!python HW8_main.py


Output dir: K114
/Users/victoriaostebo/Documents/GitHub/BIOS-584/self_py_fun/HW8Fun.py:55: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
Saved figures:
K114/Mean.png
K114/Covariance_Target.png
K114/Covariance_Non-Target.png
K114/Covariance_All.png


## Q2. A real-world data anlaysis using `Pandas` and `Scipy` (10')

<font size='4'>

- Back to the `PTSD dataset.xlsx`, let's import the dataset and name it `ptsd_df`. (no point since everyone has done it a couple of times before.)

In [88]:
# Write your own code
ptsd_df = pd.read_excel('/Users/victoriaostebo/Documents/GitHub/BIOS-584/data/PTSD dataset.xlsx')

### Q2.1. Univariate comparison (3')

<font size='4'> 
    
- Suppose that we would like to examine the utility/effect of an intervention program for patients with PTSD.
- We measure PCL5 scores at completion (`pcl5week_score.completion`) and PCL5 score at 3-month follow-up (`pcl5month_score.3_month_follow_up`). Let's assume the first score is pre-intervention and the second score is post-intervention.
- Report the summary statistics for each variable including mean, std, median, Q1, and Q3.
- Note that each patient will receive such two PCL5 scores. Use a appropriate statistical test to perform the univariate comparison. Report the outputing statistic and p-value.
- Before you run the statistic test, determine the data type and check the missingness of two columns. In particular, report the number of NA values for each variable.

In [89]:
# Write your own code
ptsd_df = pd.read_excel('/Users/victoriaostebo/Documents/GitHub/BIOS-584/data/PTSD dataset.xlsx', sheet_name='main_dataset')

#print columns
print(ptsd_df.columns)

ptsd_df = ptsd_df.rename(columns={
    'pcl5week_score.completion': 'pre_score',
    'pcl5month_score.3_month_follow_up': 'post_score'})

#check data 
print(ptsd_df['pre_score'].dtype)
print(ptsd_df['post_score'].dtype)

#check missing values
ptsd_df['pre_score'].isna().sum()
print(ptsd_df['pre_score'].isna().sum())

ptsd_df['post_score'].isna().sum()
print(ptsd_df['post_score'].isna().sum())

#summary statistics-mean 
ptsd_df['pre_score'].mean()
print(ptsd_df['pre_score'].mean())
ptsd_df['post_score'].mean()
print(ptsd_df['post_score'].mean())

#summary statistics-std
ptsd_df['pre_score'].std()
print(ptsd_df['pre_score'].std())
ptsd_df['post_score'].std()
print(ptsd_df['post_score'].std())

#summary statistics-median
ptsd_df['pre_score'].median()
print(ptsd_df['pre_score'].median())
ptsd_df['post_score'].median()
print(ptsd_df['post_score'].median())

#summary statistics-Q1
ptsd_df['pre_score'].quantile(0.25)
print(ptsd_df['pre_score'].quantile(0.25))
ptsd_df['post_score'].quantile(0.25)
print(ptsd_df['post_score'].quantile(0.25))

#summary statistics-Q3
ptsd_df['pre_score'].quantile(0.75)
print(ptsd_df['pre_score'].quantile(0.75))
ptsd_df['post_score'].quantile(0.75)
print(ptsd_df['post_score'].quantile(0.75))

#paired t-test
paired_data = ptsd_df[['pre_score', 'post_score']].dropna()
t_stat, p_val = stats.ttest_rel(paired_data['pre_score'], paired_data['post_score'])
print(t_stat)
print(p_val)


Index(['record_id', 'ptsdpresent_caps', 'caps_minuspcl', 'caps_minuspcl_code',
       'caps_intake', 'pcl5_score_intake', 'pcl5month_score.baseline',
       'pcl5week_score.completion', 'pcl5month_score.3_month_follow_up',
       'mdd_code',
       ...
       'state_code', 'va_benefits_indicator', 'wwp_alumni_indicator', 'army',
       'airforce', 'marines', 'navy', 'coastguard', 'nationalguard',
       'reserve'],
      dtype='object', length=439)
float64
float64
27
251
29.43859649122807
31.719827586206897
19.373839884545028
18.812907606503504
25.0
29.5
15.0
16.0
42.0
44.25
-4.709238006778945
4.370038747681667e-06


### Q2.2. Multiple Linear Regression (7')

<font size='4'>

- Select columns specified in the following code chunk and create a subset dataset named `ptsd_sub_df`.
- Fit a linear regression to examine the association between `caps_intake` (continuous outcome) and the remaining covariates (as predictors) using `ptsd_sub_df`.
    - Note that all covariates ending with `_code` are categorical variables.
- Use the instruction here to write the formula for linear regression in Python.
    - https://www.statsmodels.org/stable/example_formulas.html
- Report the output page including R2, adjusted R2, and parameter estimates, SE, 95% confidence intervals, and p-values.
- Provide a brief interpretation for all significant predictors (p<0.05) excluding the intercept.
- Relevant label information includes:
    - `employment_code`: 1: Employed, 2: Unemployed, 3: Retired, 4: Disabled/Unable to work, 5: Student, 6: Other.
    - `rank_code`: 1. Enlisted, 2: Officer, 3: Other

In [90]:
# The following column names are used for linear regression.
# Do not delete.
relevant_col_names = ['caps_intake', 'age_iop', 'gender_code', 'sexualorient_code', 'race_code', 'ethnicity_code', 
                      'education_code', 'employment_code',
                      'rank_code', 'branch_code', 'mdd_code', 'ctq_total_score', 'sexual_trauma', 'sud_code']



In [91]:
# Write your own code
#create subset dataset
ptsd_sub_df = ptsd_df[relevant_col_names].copy()

#covert code variables & lin reg based on example
model = smf.ols(formula='caps_intake ~ age_iop + C(gender_code) + C(sexualorient_code) + C(race_code) + C(ethnicity_code) + C(education_code) + C(employment_code) + C(rank_code) + C(branch_code) + C(mdd_code) + ctq_total_score + sexual_trauma + C(sud_code)', 
                data=ptsd_sub_df).fit()

#print the summary
print(model.summary())


#get R-squared
model.rsquared
print(model.rsquared)

model.rsquared_adj
print(model.rsquared_adj)

#get confidence intervals
conf_int = model.conf_int()

#create results table
results = pd.DataFrame({
    'Coefficient': model.params,
    'Std Error': model.bse,
    '95% CI Lower': conf_int[0],
    '95% CI Upper': conf_int[1],
    'p-value': model.pvalues})
print(results)

#get significant predictors
significant = results[results['p-value'] < 0.05]
significant = significant[significant.index != 'Intercept']
print(significant)

                            OLS Regression Results                            
Dep. Variable:            caps_intake   R-squared:                       0.114
Model:                            OLS   Adj. R-squared:                  0.051
Method:                 Least Squares   F-statistic:                     1.796
Date:                Thu, 06 Nov 2025   Prob (F-statistic):            0.00697
Time:                        21:54:15   Log-Likelihood:                -1569.0
No. Observations:                 448   AIC:                             3200.
Df Residuals:                     417   BIC:                             3327.
Df Model:                          30                                         
Covariance Type:            nonrobust                                         
                                coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------
Intercept             

In [92]:
# Write your interpretations below:

# In this model, three variables were significantly (p < 0.05) related to CAPS intake scores. 
    #First, individuals who were disabled or unable to work had higher PTSD scores than those who were employed (about +2.74 points), suggesting greater PTSD severity in this group. 
    #Second, officers had lower PTSD scores compared to enlisted service members (about −3.06 points), meaning officers showed less severe PTSD symptoms. 
    #Finally, higher childhood trauma scores were linked to slightly higher PTSD scores (about +0.04 points for each point on the CTQ scale), indicating that more childhood trauma is associated with greater PTSD severity.